In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_postgres.vectorstores import PGVector
from langchain_core.documents import Document

load_dotenv()

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

connection_url = os.getenv("DATABASE_URL")
vector_store = PGVector(
    embeddings=embedding_model,
    collection_name="integrated_university_rules",
    connection=connection_url,
    use_jsonb=True
)

# 1차 등록 대상 문서 (2026 규정 및 장학 규정)
initial_documents = [
    Document(
        page_content="[도서관 규정] 학부생은 최대 5권까지 14일간 대출 가능합니다.",
        metadata={"source": "rules_library.txt", "category": "library", "year": 2026}
    ),
    Document(
        page_content="[장학금 규정] 직전 학기 평점 3.5 이상 시 장학금 대상입니다.",
        metadata={"source": "rules_scholarship.txt", "category": "scholarship", "year": 2026}
    )
]

vector_store.add_documents(initial_documents)

['9de07ed7-28d2-4673-84e4-883e3a510c47',
 'a9e4413f-21f3-49bb-accb-023a94ddc421']

In [3]:
user_query = "학부생의 도서 대출 권수와 기간을 알려주세요."

retriever = vector_store.as_retriever()

searched_docs = retriever.invoke(user_query)

for doc in searched_docs:
    print(doc)

page_content='[도서관 규정] 학부생은 최대 5권까지 14일간 대출 가능합니다.' metadata={'year': 2026, 'source': 'rules_library.txt', 'category': 'library'}
page_content='[장학금 규정] 직전 학기 평점 3.5 이상 시 장학금 대상입니다.' metadata={'year': 2026, 'source': 'rules_scholarship.txt', 'category': 'scholarship'}


In [4]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in searched_docs)

In [6]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

model = init_chat_model(
    model_provider="openai",
    model="gpt-6-luna",
    reasoning_effort="low"
)

parser = StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system","""당신은 대학 학사 규정 안내 전문가 입니다.
    제공된 [참고 문서]에만 근거해서 답변하고, 문서에 없는 내용은 '죄송하지만 알지 못합니다.'라고 답변하세요.
    
    [참고 문서]
    {context}
    """),
    ("user", "{question}")
])

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | parser
)

response = chain.invoke("도서관 대출 권수와 기간 규정을 알려줘.")
print(response)

TypeError: BaseModel.__init__() takes 1 positional argument but 2 were given

In [ ]:
# vector_store.as_retriever() - 설정
"""
search_kwargs={
    "k" : 3, Top-k, 유사도 기준 상위 3개 추출
    "filter" : { ...}, metadata 필터링 조건...
    "score_threshold": ..., -> 필수로 
    search_type = "similiarity_score_threshold" 설정이 필수
}
"""


In [ ]:
query = "도서관 대출 권수와 기간 규정을 알려줘."
# cosine 유사도 점수는 0에 가까울수록 유사함. 작은 숫자가 위로 출력됨
searched_docs1 = vector_store.similarity_search_with_score(
    query=query
)

for doc, score in searched_docs1:
    print(f"score={score:.4f}, doc = {doc}")

score=0.4496, doc = page_content='[도서관 규정] 학부생은 최대 5권까지 14일간 대출 가능합니다.' metadata={'year': 2026, 'source': 'rules_library.txt', 'category': 'library'}
score=0.7657, doc = page_content='[장학금 규정] 직전 학기 평점 3.5 이상 시 장학금 대상입니다.' metadata={'year': 2026, 'source': 'rules_scholarship.txt', 'category': 'scholarship'}


In [13]:
# similarity_search_with_relevance 유사도 임계치를 설정할 수 있음
SCORE_THRESHOLD = 0.7
# 1에 가까울수록 유사 (0~1) 동일하게 코사인 유사도 기반, 숫자가 클수록 유사하게끔
searched_docs2 = vector_store.similarity_search_with_relevance_scores(
    query=query,
    score_threshold = 1 - SCORE_THRESHOLD # data >= 0.7
    )

for doc, score in searched_docs2:
    print(f"score={score:.4f}, doc = {doc}")


score=0.5504, doc = page_content='[도서관 규정] 학부생은 최대 5권까지 14일간 대출 가능합니다.' metadata={'year': 2026, 'source': 'rules_library.txt', 'category': 'library'}


In [10]:
retriever2 = vector_store.as_retriever(
    search_type = "similarity_score_threshold",
    search_kwargs={"score_threshold": 1- SCORE_THRESHOLD}
)

searched_docs3 = retriever2.invoke(query)

searched_docs3

[Document(id='9de07ed7-28d2-4673-84e4-883e3a510c47', metadata={'year': 2026, 'source': 'rules_library.txt', 'category': 'library'}, page_content='[도서관 규정] 학부생은 최대 5권까지 14일간 대출 가능합니다.')]